# A self-correcting analyst (multi-agent)

This is the repo's **agentic** capstone. Four roles cooperate to answer a data question:

- **Orchestrator** — decomposes the question into independent sub-analyses (a structured plan).
- **Analyst** — writes and runs Python (code-execution server tool) to answer one sub-analysis.
- **Critic** — reviews the finding and returns a pass/fail verdict; if it's weak, the analyst
  **revises** (a self-correction loop, capped).
- **Synthesizer** — combines the verified findings into a final report.

Control flow is plain Python (deterministic orchestration); the model does the reasoning.

**Requirements:** `ANTHROPIC_API_KEY` (env var or a `.env` at the repo root). This notebook makes
several API calls per run (plan + analyst/critic per sub-analysis + synthesis), so it costs more
than the single-call examples — keep the sub-analysis and revision caps low while experimenting.

## Setup

The four agents and the orchestration loop live in `_analyst.py`.

In [ ]:
import os
import sys

from dotenv import load_dotenv
from anthropic import Anthropic

for _p in (".", "agents"):
    if os.path.isfile(os.path.join(_p, "_analyst.py")) and _p not in sys.path:
        sys.path.insert(0, _p)

from _analyst import (
    DEFAULT_QUESTION,
    OBSERVATIONS_CSV,
    orchestrate,
    upload_dataset,
    write_dataset,
)

load_dotenv()
client = Anthropic()

## The dataset

A generated wildlife-survey CSV with **planted data-quality issues** (an outlier count and a
missing value) so the analyst/critic loop has something real to catch. We upload it once via the
Files API; every analyst sub-agent references the same `file_id`.

In [ ]:
path = write_dataset("observations.csv")
file_id = upload_dataset(client, path).id
print("\n".join(OBSERVATIONS_CSV.splitlines()[:6]))

## Run the multi-agent loop

`orchestrate` runs the whole thing and calls `emit` for each step, so we can watch it work:
plan → (analyst ⇄ critic) per sub-analysis → synthesize. We keep the caps small here.

In [ ]:
def emit(e):
    k = e["kind"]
    if k == "plan":
        print("🧭 PLAN:")
        for i, s in enumerate(e["subtasks"], 1):
            print(f"   {i}. {s}")
    elif k == "subtask_start":
        print(f"\n🔎 {e['subtask']}")
    elif k == "analyst":
        print(f"   🔬 analyst (attempt {e['attempt']}): {e['finding'][:110].strip()}…")
    elif k == "critic":
        print("   ✅ critic: passed" if e["passed"] else f"   ♻️ critic: {'; '.join(e['issues'])}")
    elif k == "synthesizing":
        print("\n🧩 synthesizing…")

out = orchestrate(client, DEFAULT_QUESTION, file_id, OBSERVATIONS_CSV,
                  max_subtasks=2, max_revisions=1, emit=emit)

## The final report

The synthesizer's combined answer, with any unverified findings flagged.

In [ ]:
from IPython.display import Markdown

print("verified:", [r["verdict"].passed for r in out["results"]])
Markdown(out["final"])

## Notes & extensions

- **Why multi-agent?** Decomposing into independent sub-analyses (fan-out) keeps each step focused
  and lets a dedicated **critic** catch errors the analyst won't catch about itself.
- **Self-correction** — the analyst ⇄ critic loop is the reliability mechanism. Always **cap**
  revisions (and sub-analyses) so the loop terminates.
- **Roles via prompts + structured hand-offs** — the orchestrator and critic return Pydantic
  objects (`Plan`, `Verdict`), so control flow stays clean. The analyst uses the code-execution
  *server* tool as its action.
- **Context management** — only the analyst's finding + computed output flow to the critic and
  synthesizer (not full transcripts), keeping the window small.
- **Make it stronger:** run sub-analyses in parallel (threads); let the critic *re-run code* to
  verify rather than reason over the output; add a planner that reacts to findings; cache the
  shared system prompt across the many calls.